# Splitting and merging

In [1]:
import os 
import cv2
import rasterio

import numpy as np

#from samgeo import SamGeo2

import geopandas as gpd
import pickle
from pyproj import Transformer

import matplotlib.pyplot as plt

import leafmap.leafmap as leafmap 
 
#from plantcv import plantcv as pcv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tifffile 


from utils.raster_tools import Raster_profile #class type for raster profile
from utils.tools import get_raster_data, image_enhancement, save_raster_and_write_meta
from utils.tools import setup_polygon, save_stats, read_npz

clipped_theos_file = "theos/clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif"


# Specify row and column here:
slice_row       = 10
slice_column    = 10


zoom_level      = 18 
slices_path     = "ISP0704-Zoom%d" % zoom_level
slice_subpath   = os.path.join(slices_path, "%000d-%000d" % (slice_row, slice_column)) 
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif")  
slice_theos_filename  = os.path.join(slice_subpath, "theos.tif") 


target_dir      = os.path.join(slice_subpath, "samgeo2mask")
os.makedirs(target_dir, exist_ok=True)



warped_slice_google_filename_top_left = os.path.join(slice_subpath, "warped_google_top_left.tif") 
warped_slice_google_filename_top_right = os.path.join(slice_subpath, "warped_google_top_right.tif") 
warped_slice_google_filename_bottom_left = os.path.join(slice_subpath, "warped_google_bottom_left.tif") 
warped_slice_google_filename_bottom_right = os.path.join(slice_subpath, "warped_google_bottom_right.tif") 

 

target_dir_top_left = os.path.join(slice_subpath, "samgeo2mask_top_left")
mask_filename_top_left = os.path.join(target_dir_top_left, "masks.tif")

target_dir_top_right    = os.path.join(slice_subpath, "samgeo2mask_top_right")
mask_filename_top_right = os.path.join(target_dir_top_right, "masks.tif")

target_dir_bottom_left = os.path.join(slice_subpath, "samgeo2mask_bottom_left")
mask_filename_bottom_left = os.path.join(target_dir_bottom_left, "masks.tif")

target_dir_bottom_right = os.path.join(slice_subpath, "samgeo2mask_bottom_right")
mask_filename_bottom_right = os.path.join(target_dir_bottom_right, "masks.tif")

## Splitting google image

In [ ]:
stats_          = read_npz(os.path.join(slice_subpath, "stats.npz") )
lat_start_temp  = stats_["lat_start_temp"]
long_start_temp = stats_["long_start_temp"]

lat_end_temp  = stats_["lat_end_temp"]
long_end_temp = stats_["long_end_temp"]

crs_source = "EPSG:4326" # Google 
crs_target = "EPSG:32647" # Theos

lat_end_half   = lat_start_temp - np.abs(lat_end_temp - lat_start_temp)*0.5
long_end_half  = long_start_temp + np.abs(long_end_temp - long_start_temp)*0.5

warped_slice_google_filename_top_left = os.path.join(slice_subpath, "warped_google_top_left.tif") 
poly_gons_top_left, bbox_top_left, coordinates_top_left = setup_polygon(long_start_temp, lat_start_temp, long_end_half, lat_end_half, crs_source=crs_source, crs_target=crs_target)
leafmap.clip_image(warped_slice_google_filename, poly_gons_top_left, warped_slice_google_filename_top_left)

warped_slice_google_filename_top_right = os.path.join(slice_subpath, "warped_google_top_right.tif") 
poly_gons_top_right, bbox_top_right, coordinates_top_right = setup_polygon(long_end_half, lat_start_temp, long_end_temp, lat_end_half, crs_source=crs_source, crs_target=crs_target)
leafmap.clip_image(warped_slice_google_filename, poly_gons_top_right, warped_slice_google_filename_top_right)

warped_slice_google_filename_bottom_left = os.path.join(slice_subpath, "warped_google_bottom_left.tif") 
poly_gons_bottom_left, bbox_bottom_left, coordinates_bottom_left = setup_polygon(long_start_temp, lat_end_half, long_end_half, lat_end_temp, crs_source=crs_source, crs_target=crs_target)
leafmap.clip_image(warped_slice_google_filename, poly_gons_bottom_left, warped_slice_google_filename_bottom_left)

warped_slice_google_filename_bottom_right = os.path.join(slice_subpath, "warped_google_bottom_right.tif") 
poly_gons_bottom_right, bbox_bottom_right, coordinates_bottom_right = setup_polygon(long_end_half, lat_end_half, long_end_temp, lat_end_temp, crs_source=crs_source, crs_target=crs_target)
leafmap.clip_image(warped_slice_google_filename, poly_gons_bottom_right, warped_slice_google_filename_bottom_right)


In [ ]:
warped_slice_google_filename_top_left = os.path.join(slice_subpath, "warped_google_top_left.tif") 
warped_slice_google_filename_top_right = os.path.join(slice_subpath, "warped_google_top_right.tif") 
warped_slice_google_filename_bottom_left = os.path.join(slice_subpath, "warped_google_bottom_left.tif") 
warped_slice_google_filename_bottom_right = os.path.join(slice_subpath, "warped_google_bottom_right.tif") 


m = leafmap.Map(center=[12.908807, 100.922147], zoom=18 , height="800px") 
m.add_raster(warped_slice_google_filename, layer_name="warped_google.tif") 
m.add_raster(warped_slice_google_filename_top_left, layer_name="top_left") 
m.add_raster(warped_slice_google_filename_top_right, layer_name="top_right") 
m.add_raster(warped_slice_google_filename_bottom_left, layer_name="bottom_left") 
m.add_raster(warped_slice_google_filename_bottom_right, layer_name="bottom_right") 
m

## SamGeo

### Top left

In [ ]:
target_dir_top_left = os.path.join(slice_subpath, "samgeo2mask_top_left")
mask_filename_top_left = os.path.join(target_dir_top_left, "masks.tif")
os.makedirs(target_dir_top_left, exist_ok=True)
print("PLEASE SAVE THE MASK under Folder: %s" % target_dir_top_left)

In [ ]:
from samgeo import SamGeo2
sam = SamGeo2(
    model_id="sam2-hiera-large",
    automatic=False,
    device="cuda"
) 

sam.set_image(warped_slice_google_filename_top_left)
sam.show_map()

### Top right

In [ ]:
target_dir_top_right    = os.path.join(slice_subpath, "samgeo2mask_top_right")
mask_filename_top_right = os.path.join(target_dir_top_right, "masks.tif")
os.makedirs(target_dir_top_right, exist_ok=True)
print("PLEASE SAVE THE MASK under Folder: %s" % target_dir_top_right)

In [ ]:
from samgeo import SamGeo2
sam = SamGeo2(
    model_id="sam2-hiera-large",
    automatic=False,
    device="cuda"
) 

sam.set_image(warped_slice_google_filename_top_right)
sam.show_map()

### Bottom left

In [ ]:
target_dir_bottom_left = os.path.join(slice_subpath, "samgeo2mask_bottom_left")
mask_filename_bottom_left = os.path.join(target_dir_bottom_left, "masks.tif")
os.makedirs(target_dir_bottom_left, exist_ok=True)
print("PLEASE SAVE THE MASK under Folder: %s" % target_dir_bottom_left)

In [ ]:
from samgeo import SamGeo2
sam = SamGeo2(
    model_id="sam2-hiera-large",
    automatic=False,
    device="cuda"
) 

sam.set_image(warped_slice_google_filename_bottom_left)
sam.show_map()

### Bottom right

In [ ]:
target_dir_bottom_right = os.path.join(slice_subpath, "samgeo2mask_bottom_right")
mask_filename_bottom_right = os.path.join(target_dir_bottom_right, "masks.tif")
os.makedirs(target_dir_bottom_right, exist_ok=True)
print("PLEASE SAVE THE MASK under Folder: %s" % target_dir_bottom_right)

In [ ]:
from samgeo import SamGeo2
sam = SamGeo2(
    model_id="sam2-hiera-large",
    automatic=False,
    device="cuda"
) 

sam.set_image(warped_slice_google_filename_bottom_right)
sam.show_map()

# Merging

## Check masks before merging

In [ ]:
import leafmap.leafmap as leafmap
m = leafmap.Map() 
m.add_raster(slice_theos_filename, layer_name="theos") 
m.add_raster(warped_slice_google_filename, layer_name="Google (warped)")  
m.add_raster(mask_filename_top_left, cmap="jet", layer_name="Mask (top left)")  
m.add_raster(mask_filename_top_right, cmap="jet", layer_name="Mask (top right)")  
m.add_raster(mask_filename_bottom_left, cmap="jet", layer_name="Mask (bottom left)")  
m.add_raster(mask_filename_bottom_right, cmap="jet", layer_name="Mask (bottom right)")  
m

## Merge GeoTiff 

In [ ]:
import rasterio
from utils.mask_tools import Mask_profile
from rasterio.merge import merge
import pandas as pd
import geopandas as gpd
import os
import shutil

target_dir = os.path.join(slice_subpath, "samgeo2mask")
os.makedirs(target_dir, exist_ok=True)


sub_paths = [ 
            os.path.join(slice_subpath, "samgeo2mask_top_left"), 
            os.path.join(slice_subpath, "samgeo2mask_top_right"), 
            os.path.join(slice_subpath, "samgeo2mask_bottom_left"), 
            os.path.join(slice_subpath, "samgeo2mask_bottom_right")
            ]

mask_filename_prev  = os.path.join(sub_paths[0], "masks.tif")   
shutil.copy(mask_filename_prev, os.path.join(sub_paths[0], "masks-edited.tif") )

# update object ids
for i in range(1, len(sub_paths)): 
    mask_filename_prev  =  os.path.join(sub_paths[i-1], "masks-edited.tif")    
    center_geojson_prev = os.path.join(sub_paths[i-1], "masks_fg_markers.geojson")   

    mask_filename       = os.path.join(sub_paths[i], "masks.tif") 
    center_geojson      = os.path.join(sub_paths[i], "masks_fg_markers.geojson")    

    meta_mask_filename = mask_filename


    Mask_obj_top = Mask_profile(mask_filename_prev, center_geojson_file=center_geojson_prev, center_geojson_crs="EPSG:4326")
    object_id = np.unique(Mask_obj_top.mask)
    num_obs   = max(object_id).item() 

    Mask_obj_bottom = Mask_profile(mask_filename, center_geojson_file=center_geojson, center_geojson_crs="EPSG:4326")
    mask2D = Mask_obj_bottom.mask
    mask2D[mask2D > 0] = mask2D[mask2D > 0] + num_obs

    # update mask   
    edited_mask_filename   = os.path.join(sub_paths[i], "masks-edited.tif") 
    mask_2D              = mask2D.reshape(1, mask2D.shape[0], mask2D.shape[1])
    save_raster_and_write_meta(mask_2D, edited_mask_filename, meta_mask_filename)

# merge geo tif and write output 
tif_sub_paths = [os.path.join(path, "masks-edited.tif") for path in sub_paths]


src_files = [rasterio.open(fp) for fp in tif_sub_paths]
mosaic, out_trans = merge(src_files)

out_meta = src_files[0].meta.copy()
out_meta.update({
    "driver": "GTiff", "height": mosaic.shape[1],
    "width": mosaic.shape[2], "transform": out_trans,
    "compress": "lzw" 
})


output_combined_masks_file  = os.path.join(target_dir, "masks.tif")  
with rasterio.open(output_combined_masks_file, 'w', **out_meta) as dest:
    dest.write(mosaic) 
for src in src_files: src.close() 



# merge center geopandas  
gdf_prev    = gpd.read_file(os.path.join(sub_paths[0], "masks_fg_markers.geojson")) 
for i in range(1, len(sub_paths)):
    gdf      = gpd.read_file(os.path.join(sub_paths[i], "masks_fg_markers.geojson"))
    gdf_prev = pd.concat([gdf_prev, gdf], ignore_index=True)


merged_center_filename = os.path.join(target_dir, "masks_fg_markers.geojson") 
gdf_prev.to_file(merged_center_filename, driver='GeoJSON')  

Mask_obj_bottom_edited = Mask_profile(output_combined_masks_file, center_geojson_file=merged_center_filename, center_geojson_crs="EPSG:4326")
gdf_bottom = Mask_obj_bottom_edited.make_boundboxes()
merged_bb_filename = os.path.join(target_dir, "boundbox.geojson")  
gdf_bottom.to_file(merged_bb_filename, driver='GeoJSON')  

## Check the result

In [ ]:
m = leafmap.Map()
m.add_raster(warped_slice_google_filename, layer_name="Image") 
m.add_circle_markers_from_xy(merged_center_filename, radius=3, color="red", fill_color="yellow", fill_opacity=0.8)   
m.add_raster(output_combined_masks_file, cmap="jet", layer_name="Building masks")   
m.add_vector(merged_bb_filename, layer_name="Bounding Boxes")
m